## Chapter 4 Exercise 4.1

In [1]:
from importlib.metadata import version

used_libraries = [
    "matplotlib",
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

matplotlib version: 3.11.1
reasoning_from_scratch version: 0.1.21
torch version: 2.10.0
tokenizers version: 0.22.2


In [2]:
import torch
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
     load_model_and_tokenizer
)

device = get_device()

# Use CPU for the first run of this chapter
device = torch.device("cpu")

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

Using Apple Silicon GPU (MPS)
✓ qwen3/qwen3-0.6B-base.pth already up-to-date


In [3]:
from reasoning_from_scratch.ch03 import render_prompt

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

print(prompt)

You are a helpful math assistant.
Answer the question and write the final result on a new line as:
\boxed{ANSWER}

Question:
Half the value of $3x-9$ is $x+37$. What is the value of $x$?

Answer:


In [9]:
from reasoning_from_scratch.ch02 import generate_text_basic_stream_cache
from reasoning_from_scratch.ch04 import generate_text_stream_concat_flex

In [14]:
from reasoning_from_scratch.ch03 import *

In [32]:
def evaluate_math500_cot_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=512,
    verbose=False,
):

    if out_path is None:
        dev_name = str(device).replace(":", "-")  # Make filename compatible with Windows
        out_path = Path(f"math500-{dev_name}.jsonl")

    num_examples = len(math_data)
    num_correct = 0
    total_len = 0  # Calculates the average response length (see exercise 3.2)
    start_time = time.time()

    with open(out_path, "w", encoding="utf-8") as f:  # Save results for inspection
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"])    # 1. Apply prompt template
            prompt_cot = prompt + " \n\nExplain step by step."  # 1.5 add cot prompt
            gen_text = generate_text_stream_concat(   # 2. Generate response
                model, tokenizer, prompt_cot, device,
                max_new_tokens=max_new_tokens,
                verbose=verbose,
            )
            total_len += len(tokenizer.encode(gen_text))

            extracted = extract_final_candidate(  # 3. Extract and normalize answer
                gen_text
            )
            is_correct = grade_answer(            # 4. Grade answer
                extracted, row["answer"]
            )
            num_correct += int(is_correct)

            record = {  # Record to be saved for inspection
                "index": i,
                "problem": row["problem"],
                "gtruth_answer": row["answer"],
                "generated_text": gen_text,
                "extracted": extracted,
                "correct": bool(is_correct),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

            progress_msg = eta_progress_message(
                processed=i,
                total=num_examples,
                start_time=start_time,
                show_eta=True,
                label="MATH-500",
            )
            print(progress_msg, end="\r", flush=True)
            if verbose:  # Print responses during the generation process
                print(
                    f"\n\n{'='*50}\n{progress_msg}\n"
                    f"{'='*50}\nExtracted: {extracted}\n"
                    f"Expected:  {row['answer']}\n"
                    f"Correct so far: {num_correct}\n{'-'*50}"
                )

    # Print summary information
    seconds_elapsed = time.time() - start_time
    acc = num_correct / num_examples if num_examples else 0.0
    print(f"\nAccuracy: {acc*100:.1f}% ({num_correct}/{num_examples})")
    print(f"Total time: {seconds_elapsed/60:.1f} min")
    avg_len = total_len / num_examples
    print(f"Average response length: {avg_len:.2f} tokens")
    print(f"Logs written to: {out_path}")
    return num_correct, num_examples, acc


In [17]:
math_data = load_math500_test()

In [18]:
len(math_data)

500

In [21]:
math_data[0]['problem']

'Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$'

In [29]:
test_prompt = render_prompt(math_data[0]['problem'])  + " \n\nExplain step by step."

In [28]:
generated_text = generate_text_stream_concat(
    model, tokenizer, test_prompt, device,
    max_new_tokens=2048,
    verbose=True
)

 To convert the point \((0, 3)\) from rectangular coordinates to polar coordinates, we need to find the radius \(r\) and the angle \(\theta\). Here's the step-by-step process:

### Step 1: Find the radius \(r\)
The radius \(r\) is the distance from the origin to the point \((x, y)\). It is calculated using the formula:
\[
r = \sqrt{x^2 + y^2}
\]
For the point \((0, 3)\):
\[
r = \sqrt{0^2 + 3^2} = \sqrt{9} = 3
\]
So, \(r = 3\).

### Step 2: Find the angle \(\theta\)
The angle \(\theta\) is the angle formed with the positive \(x\)-axis. It is calculated using the formula:
\[
\theta = \arctan\left(\frac{y}{x}\right)
\]
However, we need to consider the quadrant in which the point lies to determine the correct angle.

- If \(x > 0\) and \(y > 0\), the point is in the first quadrant, and \(\theta\) is the angle from the positive \(x\)-axis.
- If \(x < 0\) and \(y > 0\), the point is in the second quadrant, and \(\theta\) is \(\pi + \arctan\left(\frac{y}{x}\right)\).
- If \(x < 0\) and \(y < 

In [35]:
print("Model:", WHICH_MODEL)
print("Device:", device)
num_correct, num_examples, acc = evaluate_math500_cot_stream(
    model, tokenizer, device, 
    math_data=math_data[:10],
    max_new_tokens=2048,
    verbose=False
)

Model: base
Device: mps
MATH-500: 10/10 | ETA: 00s        
Accuracy: 50.0% (5/10)
Total time: 2.6 min
Average response length: 777.80 tokens
Logs written to: math500-mps.jsonl


The full run would take two hours, i decided NAH, but we can see on the  first 10 the accuracy was higher then the non-cot version from chapter 3

In [ ]:
print("Model:", WHICH_MODEL)
print("Device:", device)
num_correct, num_examples, acc = evaluate_math500_cot_stream(
    model, tokenizer, device, 
    math_data=math_data,
    max_new_tokens=2048,
    verbose=False
)